# **Step 1: Extract PPG Signals from MIMIC-III Matched Dataset**

## Dataset Preparation Pipeline Overview
**Step 1** → Extract raw PPG waveforms
**Step 2** → Add demographic data (age, gender)
**Step 3** → Apply disease labels (CAD/CVD)
**Step 4** → Prepare patient-level datasets

## This Step
Extracts PPG waveforms from MIMIC-III records and saves to CSV format.
**Output:** `all-ppg-data.csv` with columns (patient_id, group, record_id, fs, ppg_min_3-8)

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
WORK_DIR = PROJECT_ROOT / "work" / "mimic_waveforms"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Place the MIMIC-III matched waveform RECORDS file here before running extraction.
RECORDS_FILE = WORK_DIR / "RECORDS"
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {WORK_DIR}")

## **Import Libraries**

In [ ]:
import wfdb
import os
import numpy as np
import pandas as pd

In [ ]:
# Read patient paths from the RECORDS file.
# The RECORDS file is distributed with the MIMIC-III matched waveform database.
patient_data_from_file = []
records_file_path = RECORDS_FILE

if not records_file_path.exists():
    raise FileNotFoundError(
        f"Missing RECORDS file: {records_file_path}. Place the MIMIC-III matched waveform RECORDS file there first."
    )

with open(records_file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split("/")
            if len(parts) >= 2:
                group = parts[0]
                patient_id = parts[1]
                patient_data_from_file.append({"group": group, "patient_id": patient_id})

print(f"Loaded {len(patient_data_from_file)} patient entries from {records_file_path}")

In [ ]:
BASE_ROOT = "mimic3wdb-matched/1.0"

## **Function for Extracting Quality PPG**

In [ ]:
def extract_clean_ppg_min3_to_8(record_name, pn_dir):
    try:
        header = wfdb.rdheader(record_name, pn_dir=pn_dir)
        fs = int(header.fs)

        samples_per_min = fs * 60
        skip_samples = 2 * samples_per_min
        keep_samples = 6 * samples_per_min
        total_samples = skip_samples + keep_samples

        # find PPG channel
        ppg_index = None
        for i, sig in enumerate(header.sig_name):
            if "PLETH" in sig.upper() or "PPG" in sig.upper():
                ppg_index = i
                break

        if ppg_index is None:
            return None

        signal, _ = wfdb.rdsamp(
            record_name,
            pn_dir=pn_dir,
            channels=[ppg_index],
            sampto=total_samples
        )

        ppg = signal[:, 0]
        ppg = ppg[np.isfinite(ppg)]

        if len(ppg) < total_samples:
            return None

        # keep minute 3 → 8
        ppg = ppg[skip_samples: skip_samples + keep_samples]

        # quality checks
        if np.std(ppg) < 0.01:
            return None
        if np.ptp(ppg) < 0.05:
            return None

        return ppg, fs

    except:
        return None


## **Extract PPG Data from Available Records**

In [ ]:
import csv
import json
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache

import numpy as np
import pandas as pd
import wfdb
from tqdm.auto import tqdm

# --- Configuration ---
BASE_ROOT = "mimic3wdb-matched/1.0"
records_file_path = str(RECORDS_FILE)

# Output / resume
SAVE_OUTPUT_DIR = str(WORK_DIR / "PPG_processed_data")
SAVE_OUTPUT_FILENAME = "ppg_data_periodic.csv"
FULL_SAVE_PATH = os.path.join(SAVE_OUTPUT_DIR, SAVE_OUTPUT_FILENAME)
os.makedirs(SAVE_OUTPUT_DIR, exist_ok=True)

# Performance knobs
SAVE_FREQUENCY = 200               # larger batch = fewer disk writes
USE_PARALLEL = True                # set False to force serial mode
MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)

# Persistent cache for record lists
RECORD_CACHE_PATH = os.path.join(SAVE_OUTPUT_DIR, "record_list_cache.json")

FIELDNAMES = ["patient_id", "group", "record_id", "fs"] + [f"ppg_min_{m}" for m in range(3, 9)]


def extract_clean_ppg_min3_to_8(record_name, pn_dir):
    try:
        header = wfdb.rdheader(record_name, pn_dir=pn_dir)
        fs = int(header.fs)
        ppg_index = next(
            (i for i, s in enumerate(header.sig_name) if "PLETH" in s.upper() or "PPG" in s.upper()),
            None,
        )
        if ppg_index is None:
            return None

        samples_per_min = fs * 60
        total_samples = 8 * samples_per_min

        signal, _ = wfdb.rdsamp(
            record_name,
            pn_dir=pn_dir,
            channels=[ppg_index],
            sampto=total_samples,
        )

        ppg = signal[:, 0]
        ppg = ppg[np.isfinite(ppg)]
        if len(ppg) < total_samples:
            return None

        ppg_segment = ppg[2 * samples_per_min : 8 * samples_per_min]
        if np.std(ppg_segment) < 0.01 or np.ptp(ppg_segment) < 0.05:
            return None

        return ppg_segment, fs
    except Exception:
        return None


def minute_to_str(arr):
    # Faster than Python per-value format loops.
    return " ".join(np.char.mod("%.4f", arr.tolist()))


def load_records_cache(path):
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict):
                return data
        except Exception as e:
            print(f"Note: Could not load record-list cache ({e}). Starting fresh.")
    return {}


def save_records_cache(path, data):
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    os.replace(tmp_path, path)


def load_patient_list(path):
    if not os.path.exists(path):
        print(f"ERROR: Records file not found at {path}.")
        return []

    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("/")
            if len(p) >= 2 and p[0] and p[1]:
                out.append({"group": p[0], "patient_id": p[1]})
    return out


def load_processed_set(path):
    done = set()
    if not os.path.exists(path):
        return done

    print(f"Resuming from existing save file: {path}")
    try:
        with open(path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                g = str(row.get("group", "")).strip()
                pid = str(row.get("patient_id", "")).strip()
                if g and pid:
                    done.add((g, pid))
        print(f"Found {len(done)} already processed patients. Skipping them.")
    except Exception as e:
        print(f"Note: Could not read existing file (it might be empty or new). Error: {e}")
    return done


def append_rows_csv(path, rows):
    if not rows:
        return

    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def process_one_patient(patient_entry, get_records_fn):
    group = patient_entry["group"]
    pid = patient_entry["patient_id"]
    pn_dir = f"{BASE_ROOT}/{group}/{pid}"

    try:
        records = get_records_fn(pn_dir)
    except Exception:
        return None

    for rec in records:
        result = extract_clean_ppg_min3_to_8(rec, pn_dir)
        if not result:
            continue

        ppg_data, fs = result
        spm = fs * 60
        row = {"patient_id": pid, "group": group, "record_id": rec, "fs": fs}

        for m in range(6):
            start_idx = m * spm
            minute_slice = ppg_data[start_idx : start_idx + spm]
            row[f"ppg_min_{m+3}"] = minute_to_str(minute_slice)

        return row

    return None


# --- Load inputs / resume state ---
patient_data_from_file = load_patient_list(records_file_path)
print(f"Loaded {len(patient_data_from_file)} patient entries from {records_file_path}")

processed_patients_set = load_processed_set(FULL_SAVE_PATH)
patient_data_to_process = [
    p for p in patient_data_from_file
    if (p["group"], p["patient_id"]) not in processed_patients_set
]
print(f"Total to process now: {len(patient_data_to_process)}")

# --- Record list cache setup ---
record_list_cache = load_records_cache(RECORD_CACHE_PATH)
cache_lock = threading.Lock()
cache_dirty = False


@lru_cache(maxsize=None)
def get_record_list_cached(pn_dir):
    global cache_dirty

    with cache_lock:
        cached = record_list_cache.get(pn_dir)
    if cached is not None:
        return cached

    records = wfdb.get_record_list(pn_dir)

    with cache_lock:
        if pn_dir not in record_list_cache:
            record_list_cache[pn_dir] = records
            cache_dirty = True
    return records


# --- Main extraction loop ---
rows_buffer = []
saved_this_session = 0

if USE_PARALLEL and len(patient_data_to_process) > 1:
    print(f"Running in parallel mode with {MAX_WORKERS} workers...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [
            ex.submit(process_one_patient, p, get_record_list_cached)
            for p in patient_data_to_process
        ]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing", unit="patient"):
            row = future.result()
            if row is None:
                continue

            rows_buffer.append(row)
            saved_this_session += 1

            if len(rows_buffer) >= SAVE_FREQUENCY:
                append_rows_csv(FULL_SAVE_PATH, rows_buffer)
                print(f"\n[Auto-Save] Appended {len(rows_buffer)} entries to {FULL_SAVE_PATH}.")
                rows_buffer = []
else:
    print("Running in serial mode...")
    for patient_entry in tqdm(patient_data_to_process, desc="Processing", unit="patient"):
        row = process_one_patient(patient_entry, get_record_list_cached)
        if row is None:
            continue

        rows_buffer.append(row)
        saved_this_session += 1

        if len(rows_buffer) >= SAVE_FREQUENCY:
            append_rows_csv(FULL_SAVE_PATH, rows_buffer)
            print(f"\n[Auto-Save] Appended {len(rows_buffer)} entries to {FULL_SAVE_PATH}.")
            rows_buffer = []

# Preserve last in-memory batch for compatibility with downstream notebook cells.
rows = rows_buffer.copy()

# Final flush
if rows_buffer:
    append_rows_csv(FULL_SAVE_PATH, rows_buffer)
    print(f"\n[Auto-Save] Appended {len(rows_buffer)} entries to {FULL_SAVE_PATH}.")

if cache_dirty:
    save_records_cache(RECORD_CACHE_PATH, record_list_cache)
    print(f"Record-list cache updated: {RECORD_CACHE_PATH}")

print(
    f"\nDone. Saved this session: {saved_this_session}. "
    f"Total in file: {len(processed_patients_set) + saved_this_session}"
)

## **Save Extracted PPG Data to CSV**

In [ ]:
import pandas as pd
import os

# 1. Build the final DataFrame from the full persisted periodic file.
# This avoids saving only the last in-memory buffer from the extraction loop.
out = str(PROJECT_ROOT / "data" / "intermediate")
os.makedirs(out, exist_ok=True)

if os.path.exists(FULL_SAVE_PATH):
    df = pd.read_csv(FULL_SAVE_PATH)
    data_source = FULL_SAVE_PATH
else:
    # Fallback for fresh runs where periodic file does not exist yet.
    df = pd.DataFrame(rows)
    data_source = "in-memory rows"

# 2. Save the full extraction data to final output CSV.
OUTPUT = os.path.join(out, "all-ppg-data.csv")
df.to_csv(OUTPUT, index=False)

# --- VERIFICATION LOGIC ---
print("--- EXTRACTION VERIFICATION ---")
print(f"Data source for final CSV   : {data_source}")
print(f"Rows written to final CSV   : {len(df)}")

if not df.empty and "ppg_min_3" in df.columns and "fs" in df.columns:
    # Check the first row as a sample.
    sample_row = df.iloc[0]
    sample_count = len(str(sample_row["ppg_min_3"]).split())
    expected_count = int(sample_row["fs"]) * 60

    print(f"Verification for Patient {sample_row['patient_id']}:")
    print(f"- Sampling Rate (fs): {sample_row['fs']} Hz")
    print(f"- Expected samples per minute: {expected_count}")
    print(f"- Actual samples found in CSV column: {sample_count}")

    if sample_count == expected_count:
        print("SUCCESS: Data extraction alignment verified.")
    else:
        print("WARNING: Sample count mismatch detected.")

print("\nSUMMARY")
print("Total in RECORDS file     :", len(patient_data_from_file))
print("Already done (prior runs) :", len(patient_data_from_file) - len(patient_data_to_process))
print("Attempted this session    :", len(patient_data_to_process))
print("Successfully saved        :", saved_this_session)
print("Failed / no PPG / skipped :", len(patient_data_to_process) - saved_this_session)
print("CSV saved to              :", OUTPUT)

In [ ]:
# Optional extraction summary.
csv_path = WORK_DIR / "PPG_processed_data" / "ppg_data_periodic.csv"
if csv_path.exists():
    saved_df = pd.read_csv(csv_path)
    saved_patients = set(zip(saved_df["group"], saved_df["patient_id"]))
    unique_patients = len(saved_df[["group", "patient_id"]].drop_duplicates())
    total_in_records = len(patient_data_from_file)
    all_patients = set((p["group"], p["patient_id"]) for p in patient_data_from_file)
    untried_or_failed = all_patients - saved_patients

    print("=== SKIP LOGIC & FAILURE ANALYSIS ===")
    print(f"Total patients in RECORDS file      : {total_in_records}")
    print(f"Successfully saved patients         : {unique_patients}")
    print(f"Untried or failed patients          : {len(untried_or_failed)}")
    print(f"Total rows in periodic CSV          : {len(saved_df)}")
else:
    print(f"No periodic extraction CSV found at {csv_path}")